In [1]:
import pyActigraphy
import plotly.graph_objects as go
import os
import pandas as pd
import numpy as np
import os

In [2]:
fpath = 'C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara'

In [4]:
raw = pyActigraphy.io.read_raw_mtn(fpath + '\\Joined_21-09-22_25-08-24.mtn')

In [ ]:
raw.name

In [ ]:
raw.start_time

In [ ]:
raw.duration()

In [ ]:
raw.frequency

VISUALISING

In [ ]:
layout = go.Layout(
    title="Actigraphy data",
    xaxis=dict(title="Date time"),
    yaxis=dict(title="Counts/period"),
    showlegend=False
)
go.Figure(data=[go.Scatter(x=raw.data.index.astype(str), y=raw.data)], layout=layout)

MASKING INACTIVE DATA

In [ ]:
raw.light.create_light_mask()

In [ ]:
# manually adding mask to more than one period
periods = [{'start': '2022-12-28 12:00:00', 'stop': '2022-12-29 12:00:00'},{'start': '2023-03-04 22:40:00', 'stop': '2023-04-18 10:00:00'},{'start': '2023-05-14 22:30:00', 'stop': '2023-05-15 07:15:00'},
           {'start': '2023-05-16 09:00:00', 'stop': '2023-06-29 13:00:00'},{'start': '2023-07-31 19:15:00', 'stop': '2023-08-16 12:00:00'},{'start': '2023-08-18 10:00:00', 'stop': '2023-08-22 13:00:00'},
           ]
for period in periods:
    raw.add_mask_period(start=period['start'], stop=period['stop'])

In [ ]:
# visualising mask:
go.Figure(data=go.Scatter(x=raw.mask.index.astype(str),y=raw.mask),layout=layout)

In [ ]:
# applying the mask:
raw.light.apply_mask = True

DAILY ACTIVITY PROFILE (create one for UK and one for Italy?)

In [ ]:
layout.update(title="Daily activity profile",xaxis=dict(title="Date time"), showlegend=False);

In [ ]:
daily_profile = raw.average_daily_activity(freq='15min', cyclic=False, binarize=False)

In [ ]:
go.Figure(data=[
    go.Scatter(x=daily_profile.index.astype(str), y=daily_profile)
], layout=layout)

In [ ]:
# Activity onset:
raw.AonT(freq='15min', binarize=True)

In [ ]:
# Activity offset:
raw.AoffT(freq='15min', binarize=True)

DETECTING SLEEP PERIODS

In [ ]:
# all algorithms return a binary time serie

In [ ]:
layout = go.Layout(title="Rest/Activity detection",xaxis=dict(title="Date time"), yaxis=dict(title="Counts/period"), showlegend=False)

In [ ]:
roenneberg = raw.Roenneberg()
roenneberg_thr = raw.Roenneberg(threshold=0.25, min_seed_period='15min') # TRESHOLD = fraction of the trend to use as a treshold for sleep/wake categorisation. min_seed_period = minimum time period required to identify a potential sleep onset

Roenneberg algorithm = detects consolidated periods of similar activity patterns. It's treshold-based but uses correlations with test series of various lengths to find the consolidated period that best matches the data

In [ ]:
go.Figure(data=[
    go.Scatter(x=raw.data.index.astype(str),y=raw.data, name='Data'),
    go.Scatter(x=roenneberg.index.astype(str),y=roenneberg, yaxis='y2', name='Roenneberg'),
    go.Scatter(x=roenneberg_thr.index.astype(str),y=roenneberg_thr, yaxis='y2', name='Roenneberg (Thr:0.25)')
], layout=layout)

In [ ]:
crespo = raw.Crespo()

In [ ]:
aot = raw.Crespo_AoT()
aot

In [ ]:
# df with the onset and offset of the activity period scored by the Crespo algorithm 
sleep_periodsX = pd.DataFrame({'start': aot[0], 'stop': aot[1]})

In [ ]:
sleep_periodsX = sleep_periodsX[['stop', 'start']]
sleep_periodsX.rename(columns={'start': 'end_datetime', 'stop': 'start_datetime'}, inplace=True)
sleep_periodsX

COLE-KRIPKE ALGORITHM = epoch-by-epoch rest-activity scoring. Uses a rolling window over the data and convolute them with a "gaussian"-like kernel. If the resulting data is above a predefined treshold, classify as activity

In [ ]:
CK = raw.CK()

In [ ]:
# dataframe to store the CK sleep scoring
df_CK = pd.DataFrame({'datetime': CK.index, 'scoring': CK})

df_CK = df_CK.reset_index(drop=True)

In [ ]:
# list to store the wake counts
wake_counts = []

# function to count the number of wake periods in a sleep period
for _, row in sleep_periodsX.iterrows():
    start, end = row['start_datetime'], row['end_datetime']
    
    # filter the data to the sleep period
    sleep_period = df_CK[(df_CK['datetime'] >= start) & (df_CK['datetime'] <= end)]
    
    wake_count = sleep_period[sleep_period['scoring'] == 0].shape[0]
    
    wake_counts.append({'start_datetime': start, 'end_datetime': end, 'epochs_awake_count': wake_count})

wake_counts_df = pd.DataFrame(wake_counts)

wake_counts_df.head()

In [ ]:
# wake duration in minutes
wake_counts_df['WASO'] = wake_counts_df['epochs_awake_count']

# remove epochs_awake_count column
wake_counts_df = wake_counts_df.drop(columns='epochs_awake_count')

wake_counts_df.head()

In [ ]:
wake_counts_df.to_csv('1a_wake_counts.csv')

LIGHT DATA

In [ ]:
raw.light

In [ ]:
raw.light.data.head(1)

In [ ]:
raw.light.data.tail(1)

In [ ]:
# LIGHT + ACTIVITY
layout = go.Layout(
    xaxis=dict(title="Date time"),
    yaxis=dict(title="Activity counts/period"),
    yaxis2=dict(title='Light intensity',overlaying='y',side='right'),
    showlegend=True
)

fig1 = go.Figure([
    go.Scatter(
        x=raw.data.index.astype(str),
        y=raw.data,
        name='Activity'),
    go.Scatter(
        x=raw.light.get_channel('whitelight').index.astype(str),
        y=raw.light.get_channel('whitelight'),
        yaxis='y2', opacity=0.5,
        name='Light')
], layout=layout)

fig1.show()

In [ ]:
raw.light.create_light_mask()

In [ ]:
raw.light.add_light_mask_period(start='2023-03-04 23:00:00', stop='2023-04-18 10:00:00')
raw.light.add_light_mask_period(start='2023-05-16 09:00:00', stop='2023-06-29 13:00:00')
raw.light.add_light_mask_period(start='2023-07-31 18:00:00', stop='2023-08-16 12:00:00')
raw.light.add_light_mask_period(start='2023-08-18 10:00:00', stop='2023-08-22 12:00:00')
raw.light.add_light_mask_period(start='2024-07-31 08:00:00', stop='2024-08-04 20:00:00')

In [ ]:
raw.light.apply_mask = True

In [ ]:
layout = go.Layout(
    xaxis=dict(title="Date time"),
    yaxis=dict(title="Activity counts/period"),
    yaxis2=dict(title='Mask',overlaying='y',side='right'),
    showlegend=True
)

In [ ]:
fig2 = go.Figure([
    go.Scatter(
        x=raw.light.get_channel('whitelight').index.astype(str),
        y=raw.light.get_channel('whitelight'),
        name='whitelight'),
    go.Scatter(
        x=raw.light.mask.index.astype(str),
        y=raw.light.mask.loc[:,'whitelight'],
        yaxis='y2', opacity=0.5,
        name='Mask')
], layout=layout)

In [ ]:
fig2.show()

Light data are (log10+1)-transformed in pyActigraphy. Therefore, when threshold values are meant to applied to the light levels, the corresponding value on the (log10+1) scale should be applied.

NB: an offset of 1 is added to the light data before log10 transformation in order to avoid a divergence of the log10 function when the light data values are zero: $:nbsphinx-math:log_{10}(0) = -:nbsphinx-math:`infty `$.

So, on a (log10+1) scale:

10 lux threshold correspond to a value of log(10+1)~1

100 lux threshold correspond to a value of log(100+1)~2

1000 lux threshold correspond to a value of log(1000+1)~3

…

To simply get the exact value, use the np.log10 function of the numpy package we imported earlier:

In [ ]:
np.log10(10+1), np.log10(100+1), np.log10(1000+1)

_exposure level_

In [ ]:
help(raw.light.light_exposure_level)

In [ ]:
raw.light.light_exposure_level(agg='median')

In [ ]:
raw.light.light_exposure_level(
    threshold=1, # on a log10 scale, 2 means 10^2 and in lux it is 10^2 lux
    start_time='8:00',
    stop_time='18:00'
)

_Summary statistics (mean, median, s.d., max., min., sum)_

In [ ]:
help(raw.light.summary_statistics_per_time_bin)

In [ ]:
a = raw.light.summary_statistics_per_time_bin()

In [ ]:
raw.light.summary_statistics_per_time_bin(bins='12h')

In [ ]:
raw.light.summary_statistics_per_time_bin(
    bins=[
        ['2023-09-19 12:00:00','2023-09-19 19:59:00'],
        ['2023-09-23 12:00:00','2023-09-23 23:59:00']
    ],
    agg_func=['mean','std']
)

_Time above threshold (TAT)_

In [ ]:
help(raw.light.TAT)

In [ ]:
raw.light.TAT() # the results are identical to the total number of epochs in the recording.

In [ ]:
raw.light.TAT(threshold=2) # with a threshold of 2 this means that the light intensity must be greater than 10^2 lux to be considered as light exposure.

In [ ]:
# The parameter oformat defines the output format. Available formats: * ‘minute’:
raw.light.TAT(threshold=2, oformat='minute') 

In [ ]:
raw.light.TAT(threshold=2, oformat='timedelta') # the output is a timedelta object

In [ ]:
# Time spent above threshold at specific time periods:
raw.light.TAT(
    threshold=2, start_time='08:00:00', stop_time='20:00:00', oformat='timedelta'
)

_Time above threshold per period (TATp)_

In [ ]:
help(raw.light.TATp)

In [ ]:
raw.light.TATp(threshold=2, oformat='timedelta')

_Mean light timing (MLit)_

In [ ]:
help(raw.light.MLiT)

In [ ]:
# The light exposure data are converted to log10+1.
raw.light.MLiT(threshold=np.log10(500+1))

In [ ]:
# To convert this number of minutes into a time of day, simply divide it by the number of minutes per hour:
divmod(812.33715,60) # The mean light timing for xxx light exposure above 500 lux is around h, min for this recording.

In [ ]:
raw.light.MLiTp(threshold=np.log10(500+1))

In [ ]:
mlitp_value = raw.light.MLiTp(threshold=np.log10(500 + 1))

# Convert into hours and minutes
h, min = divmod(mlitp_value, 60)

_L5 and M10 (values and timing)_

In [ ]:
help(raw.light.LMX)

In [ ]:
# Remove the NaN values
#cleaned_data = raw.light.data.dropna()

# Update the raw.light object with the cleaned data
#raw.light._data = cleaned_data

In [ ]:
raw.light.LMX(length='10h',lowest=False)

In [ ]:
raw.light.LMX(length='5h',lowest=True)

In [ ]:
dlp = raw.light.average_daily_profile(
    rsfreq='60min', cyclic=False,# rsfreq = resampling frequency (in minutes) to compute the average daily profile 
    channel='whitelight', # channel to use for the computation
)

In [ ]:
dlp_fig = go.Figure(go.Scatter(x=dlp.index.astype(str),y=dlp))

In [ ]:
dlp_fig.show()


_Access to raw and thresholded data_

In [ ]:
b = raw.light.data

In [ ]:
# add a column with the year of each timestamp
b['year'] = b.index.year

In [ ]:
# create a dataframe for each year
b_2022 = b[b['year'] == 2022]
b_2023 = b[b['year'] == 2023]
b_2024 = b[b['year'] == 2024]

In [ ]:
raw.light.VAT(2)